# AI R&D Assignment - Curve Fitting

First, I'm just going to load up the `xy_data.csv` file and plot it to see what the curve actually looks like before jumping into the math.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution, minimize
from scipy.spatial import cKDTree

# load the data
df = pd.read_csv('data/xy_data.csv')
x_vals = df.iloc[:, 0].values
y_vals = df.iloc[:, 1].values

plt.figure(figsize=(8, 5))
plt.scatter(x_vals, y_vals, s=2, c='blue', alpha=0.6)
plt.title('Raw Data Points')
plt.show()

### Given Problem


We are given these parametric equations:
$$ x(t) = t \cos(\theta) - e^{M|t|} \sin(0.3t) \sin(\theta) + X $$
$$ y(t) = 42 + t \sin(\theta) + e^{M|t|} \sin(0.3t) \cos(\theta) $$

The main issue here is that the CSV gives us (x,y) coordinates, but it doesn't tell us the `t` value for each point. We don't even know if the rows are in order!.

In [ ]:
# the constraints given in the pdf
bounds = [
    (0, 50),      # theta
    (-0.05, 0.05), # M
    (0, 100)      # X
]

def get_curve_points(t, theta_deg, M, X):
    # convert theta to radians first
    th = np.radians(theta_deg)
    
    x = t * np.cos(th) - np.exp(M * np.abs(t)) * np.sin(0.3 * t) * np.sin(th) + X
    y = 42 + t * np.sin(th) + np.exp(M * np.abs(t)) * np.sin(0.3 * t) * np.cos(th)
    return x, y

def generate_dense_curve(theta, M, X):
    # sample 1500 points between t=6 and t=60
    t_dense = np.linspace(6, 60, 1500)
    return get_curve_points(t_dense, theta, M, X)

### Finding the Best Parameters

Since calculating the nearest point for every single coordinate creates a really bumpy (non-convex) loss landscape, a normal optimizer will probably get stuck in a local minimum. 

I'm going to use `differential_evolution` which is a global optimizer to find the general area of the best parameters, and then I'll clean it up with `Nelder-Mead` to get the exact values.

In [ ]:
def loss_function(params):
    theta, M, X = params
    cx, cy = generate_dense_curve(theta, M, X)
    
    # build a kdtree from the generated curve points
    tree = cKDTree(np.column_stack([cx, cy]))
    
    # find distance from every real data point to the closest curve point
    distances, _ = tree.query(np.column_stack([x_vals, y_vals]))
    return np.sum(distances)

print("Running differential evolution (this might take a few seconds)...")
global_res = differential_evolution(loss_function, bounds, seed=42, maxiter=300)

print("Polishing the results...")
final_res = minimize(loss_function, global_res.x, bounds=bounds, method='Nelder-Mead')

best_theta, best_M, best_X = final_res.x
print("\nDone! Here are the extracted variables:")
print(f"Theta: {best_theta:.4f} degrees")
print(f"M: {best_M:.6f}")
print(f"X: {best_X:.4f}")

### Final Verification
Just to be absolutely sure, let's plot our new fitted curve directly on top of the original data to see if it lines up.

In [ ]:
final_x, final_y = generate_dense_curve(best_theta, best_M, best_X)

plt.figure(figsize=(10, 6))
plt.scatter(x_vals, y_vals, s=10, c='lightgray', label='Original Data')
plt.plot(final_x, final_y, c='red', linewidth=2, label='My Fitted Curve')
plt.legend()
plt.show()